<a href="https://colab.research.google.com/github/chiptune234u-lgtm/SandProject/blob/main/test_sand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title ⏳ サンドボックス・エディター（高速化版）

#@markdown ### 1. サイクル設定
CYCLE_DURATION = "20s" #@param ["20s", "30s"]

# ここでDURATIONを確定
if CYCLE_DURATION == "30s":
    DURATION = 30
elSe:
    DURATION = 20

#@markdown ### 2. 基本設定
MODE = "sand"
FPS = 30 # 高速化したのでFPSを少し上げても大丈夫です
BLOCK_SIZE = 12

#@markdown ### 3. 演出設定
PREFILL_HEIGHT = 0.4

#@markdown ### 4. 風設定
WIND_STRENGTH = 0.3

import numpy as np
import cv2
import os
import glob
import random
from google.colab import drive
import colorsys
from numba import jit

# ドライブマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

WIDTH, HEIGHT = 720, 1280
COLS, ROWS = WIDTH // BLOCK_SIZE, HEIGHT // BLOCK_SIZE

# --- 動的な色生成ロジック ---
BASE_HUE = random.random()

def get_dynamic_color():
    h = (BASE_HUE + random.uniform(-0.05, 0.05)) % 1.0
    s = random.uniform(0.5, 0.8)
    v = random.uniform(0.8, 1.0)
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return [int(r*255), int(g*255), int(b*255)]

def get_contrasting_color(base_hue):
    # 補色（反対色）の計算
    contrasting_h = (base_hue + 0.5) % 1.0
    # 彩度と明度は高めに設定して目立たせる
    s = 0.9 # 高彩度
    v = 1.0 # 高明度
    r, g, b = colorsys.hsv_to_rgb(contrasting_h, s, v)
    return [int(r*255), int(g*255), int(b*255)]

# 背景準備
bg_dir = "/content/drive/MyDrive/Colab Notebooks/画像素材"
bg_files = glob.glob(os.path.join(bg_dir, "*"))
if bg_files:
    bg_img = cv2.imread(random.choice(bg_files))
    bg_img = cv2.resize(bg_img, (WIDTH, HEIGHT))
    bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
else:
    bg_img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

# --- フォント選択ロジック ---
# フォント素材フォルダのパス
font_dir = "/content/drive/MyDrive/Colab Notebooks/font_material"

# .ttfファイルをすべて取得
font_files = glob.glob(os.path.join(font_dir, "*.ttf"))

# フォントファイルが見つかった場合、ランダムに一つ選択
if font_files:
    RANDOM_FONT_PATH = random.choice(font_files)
    print(f"ランダムに選択されたフォントパス: {RANDOM_FONT_PATH}")
elSe:
    RANDOM_FONT_PATH = None
    print("フォントファイルが見つかりませんでした。font_dirを確認してください。")

# --- 音源選択ロジック ---
# 音源素材フォルダのパス
sound_dir = "/content/drive/MyDrive/Colab Notebooks/sound_material"

# 音源ファイルをすべて取得 (一般的な拡張子を考慮)
sound_files = glob.glob(os.path.join(sound_dir, "*.*")) # 任意の拡張子

# 音源ファイルが見つかった場合、ランダムに一つ選択
if sound_files:
    RANDOM_SOUND_PATH = random.choice(sound_files)
    print(f"ランダムに選択された音源パス: {RANDOM_SOUND_PATH}")
elSe:
    RANDOM_SOUND_PATH = None
    print("音源ファイルが見つかりませんでした。sound_dirを確認してください。")

# --- グリッド初期化（天井に砂をセット） ---
grid = np.zeros((ROWS, COLS, 3), dtype=np.uint8)
fill_limit = int(ROWS * PREFILL_HEIGHT)

for y in range(0, fill_limit):
    width_at_y = int((fill_limit - y) * 1.5)
    center = COLS // 2
    # NumPyのスライス代入で高速化
    start_x = max(0, center - width_at_y)
    end_x = min(COLS, center + width_at_y)
    if end_x > start_x:
        # ランダムなマスク生成
        mask = np.random.random(end_x - start_x) > 0.1
        # 色を生成して適用（少し簡易化して高速化）
        colors = np.array([get_dynamic_color() for _ in range(np.sum(mask))])
        if len(colors) > 0:
            grid[y, start_x:end_x][mask] = colors

print(f"初期化完了: モード={DURATION}s, 天井砂セット完了！")

Mounted at /content/drive
初期化完了: モード=20s, 天井砂セット完了！


In [ ]:
velocity_grid = np.zeros((ROWS, COLS, 2), dtype=np.float32)

# 爆発計算をベクトル化（Pythonループを排除して高速化）
def apply_deep_explosion(current_grid, v_grid, power=30):
    # 砂がある座標を取得
    active_mask = np.any(current_grid > 0, axis=2)
    active_y, active_x = np.where(active_mask)

    if len(active_y) == 0:
        return v_grid, (COLS//2, ROWS//2)

    min_x, max_x = np.min(active_x), np.max(active_x)
    explosion_x = random.randint(min_x, max_x)

    min_y, max_y = np.min(active_y), np.max(active_y)
    explosion_y = int(max_y - (max_y - min_y) * 0.2)

    # メッシュグリッドで距離計算を一括処理
    y_indices, x_indices = np.indices((ROWS, COLS))
    dy = y_indices - explosion_y
    dx = x_indices - explosion_x
    dist = np.sqrt(dx**2 + dy**2) + 0.1 # 0除算防止

    # 爆発範囲内のマスク作成
    explosion_mask = (dist < 60) & active_mask

    if np.any(explosion_mask):
        force = power / dist[explosion_mask]
        v_grid[explosion_mask, 0] += (dx[explosion_mask] / dist[explosion_mask]) * force * 15
        v_grid[explosion_mask, 1] += (dy[explosion_mask] / dist[explosion_mask]) * force * 15

    return v_grid, (explosion_x, explosion_y)

# Numbaで風計算を高速化
@jit(nopython=True)
def apply_wind_numba(v_grid, sand_mask, wind_force):
    # sand_maskは (ROWS, COLS) のbool配列
    rows, cols = v_grid.shape[:2]
    for y in range(rows - 1): # 最下段は除外
        for x in range(cols):
            if sand_mask[y, x]:
                v_grid[y, x, 0] += wind_force
    return v_grid

def apply_wind(current_grid, v_grid, wind_force):
    sand_mask = np.any(current_grid > 0, axis=2)
    return apply_wind_numba(v_grid, sand_mask, wind_force)

# 物理更新ロジック（既存のベクトル化を維持しつつ整理）
def update_physics_vectorized(current_grid, v_grid, mode):
    new_grid = np.zeros_like(current_grid)
    new_v_grid = np.zeros_like(v_grid)

    # 下から上へスキャン
    for y in range(ROWS - 1, -1, -1):
        # その行に砂があるかチェック（高速化のための早期リターン）
        if not np.any(current_grid[y]): continue

        sand_mask_in_row = np.any(current_grid[y, :, :], axis=1)
        sand_x_coords = np.where(sand_mask_in_row)[0]

        colors = current_grid[y, sand_x_coords]
        velocities = v_grid[y, sand_x_coords]

        vx, vy = velocities[:, 0], velocities[:, 1]

        # 移動先計算
        proposed_nx = np.clip((sand_x_coords + vx).astype(int), 0, COLS - 1)
        proposed_ny = (y + vy + 1).astype(int)

        new_vx, new_vy = vx * 0.9, vy * 0.9 # 減衰

        # --- 境界チェック ---
        falling_off_mask = proposed_ny >= ROWS

        # 画面外へ落ちる処理
        if np.any(falling_off_mask):
            target_x_fall = proposed_nx[falling_off_mask]
            # 最下段に固定
            new_grid[ROWS-1, target_x_fall] = colors[falling_off_mask]
            new_v_grid[ROWS-1, target_x_fall] = 0.0

        # 画面内での移動処理
        remaining_mask = ~falling_off_mask
        if not np.any(remaining_mask): continue

        sand_x_rem = sand_x_coords[remaining_mask]
        colors_rem = colors[remaining_mask]
        p_nx = proposed_nx[remaining_mask]
        p_ny = proposed_ny[remaining_mask]
        n_vx, n_vy = new_vx[remaining_mask], new_vy[remaining_mask]

        # Ensure p_ny is within valid row bounds
        p_ny = np.clip(p_ny, 0, ROWS - 1)

        # 衝突判定（同じ場所に行こうとする砂の競合解決）
        target_indices_linear = p_ny * COLS + p_nx
        unique_indices, first_indices = np.unique(target_indices_linear, return_index=True)

        # 移動先が空いているか確認
        # unique_indices から y, x を復元
        u_y, u_x = np.divmod(unique_indices, COLS)
        can_move = np.all(new_grid[u_y, u_x] == 0, axis=1)

        valid_indices = first_indices[can_move]

        # 移動確定
        if len(valid_indices) > 0:
            final_y = p_ny[valid_indices]
            final_x = p_nx[valid_indices]
            new_grid[final_y, final_x] = colors_rem[valid_indices]
            new_v_grid[final_y, final_x, 0] = n_vx[valid_indices]
            new_v_grid[final_y, final_x, 1] = n_vy[valid_indices]

        # 移動できなかった砂（スライド処理へ）
        # 元のインデックス配列のうち、valid_indicesに含まれないものが移動失敗
        # ここは簡易化のために元の位置に残すか、ランダムスライド
        mask_moved = np.zeros(len(sand_x_rem), dtype=bool)
        mask_moved[valid_indices] = True
        stuck_indices = np.where(~mask_moved)[0]

        if len(stuck_indices) > 0:
             for i in stuck_indices:
                curr_x = sand_x_rem[i]
                curr_c = colors_rem[i]

                # 左右どちらかに滑る
                slide = random.choice([-1, 1])
                tx = curr_x + slide
                ty = y + 1

                if 0 <= tx < COLS and ty < ROWS and np.all(new_grid[ty, tx] == 0):
                    new_grid[ty, tx] = curr_c
                    new_v_grid[ty, tx] = [n_vx[i] * 0.5, 0]
                else:
                    new_grid[y, curr_x] = curr_c
                    new_v_grid[y, curr_x] = [0, 0]

    return new_grid, new_v_grid

In [ ]:
import time
import math

def get_contrasting_color(hue):
    if 0.2 < hue < 0.8: return (255, 255, 255)
    else: return (220, 255, 220)

total_frames = FPS * DURATION
frames_out = []
ratio = DURATION / 20.0
EXPLOSION_FRAMES = [0, int(FPS * 15 * ratio)]
EMIT_STOP_F = int(FPS * 17 * ratio)
ROT_SEC = 2.5
ROTATION_START_F = total_frames - int(FPS * ROT_SEC)
WIND_MAX_STRENGTH, WIND_START_F, WIND_PEAK_F, WIND_END_F = WIND_STRENGTH, FPS * 7, FPS * 10, FPS * 13
WIND_DIRECTION = random.choice([-1, 1])

wind_forces = np.zeros(total_frames)
t_in, t_out = np.arange(WIND_START_F, WIND_PEAK_F), np.arange(WIND_PEAK_F, WIND_END_F)
wind_forces[WIND_START_F:WIND_PEAK_F] = WIND_MAX_STRENGTH * WIND_DIRECTION * np.sin((t_in - WIND_START_F) / (WIND_PEAK_F - WIND_START_F) * np.pi / 2)
wind_forces[WIND_PEAK_F:WIND_END_F] = WIND_MAX_STRENGTH * WIND_DIRECTION * np.cos((t_out - WIND_PEAK_F) / (WIND_PEAK_F - WIND_END_F) * np.pi / 2)

# --- テキスト設定（フォント変更、影の追加） ---
font = cv2.FONT_HERSHEY_COMPLEX # より角張ったクラシックなフォントに変更
font_scale_line1, font_scale_line2 = 1.1, 2.5
line_height = 95 # 行間を調整
base_y, base_x = int(HEIGHT * 0.6), 80
text_foreground_color = get_contrasting_color(BASE_HUE)
ALPHA = 0.9

# 影の設定
shadow_offset = 5 # ピクセルオフセット
shadow_color = (0, 0, 0) # 黒

print(f"🚀 {DURATION}秒ループ 生成開始...")
start_time = time.time()

for f in range(total_frames):
    current_wind_force = wind_forces[f]
    if f in EXPLOSION_FRAMES: velocity_grid, _ = apply_deep_explosion(grid, velocity_grid)
    if f < EMIT_STOP_F:
        for sx in np.random.randint(COLS // 2 - 10, COLS // 2 + 11, 3):
            if not np.any(grid[0, sx]): grid[0, sx] = get_dynamic_color()

    if abs(current_wind_force) > 0.001: velocity_grid = apply_wind(grid, velocity_grid, current_wind_force)
    grid, velocity_grid = update_physics_vectorized(grid, velocity_grid, MODE)

    sand_layer = cv2.resize(grid, (WIDTH, HEIGHT), interpolation=cv2.INTER_NEAREST)
    if f >= ROTATION_START_F:
        p = (f - ROTATION_START_F) / (total_frames - ROTATION_START_F)
        M = cv2.getRotationMatrix2D((WIDTH // 2, HEIGHT // 2), p * p * (3 - 2 * p) * 180, 1.0)
        sand_layer = cv2.warpAffine(sand_layer, M, (WIDTH, HEIGHT), flags=cv2.INTER_NEAREST)

    frame = bg_img.copy()
    mask = np.any(sand_layer > 0, axis=2)
    frame[mask] = sand_layer[mask]

    # --- カウントダウンテキスト（影付き） ---
    if f < ROTATION_START_F:
        next_events = [ef for ef in EXPLOSION_FRAMES if ef > f]
        if next_events:
            text_l1, text_l2 = "NEXT BOOM", f"{((next_events[0] - f) / FPS):.1f}s"

            overlay = frame.copy()

            # Line 1: Shadow
            cv2.putText(overlay, text_l1, (base_x + shadow_offset, base_y + shadow_offset), font, font_scale_line1, shadow_color, 8, cv2.LINE_AA)
            # Line 1: Foreground
            cv2.putText(overlay, text_l1, (base_x, base_y), font, font_scale_line1, text_foreground_color, 4, cv2.LINE_AA)

            # Line 2: Shadow
            cv2.putText(overlay, text_l2, (base_x + shadow_offset, base_y + line_height + shadow_offset), font, font_scale_line2, shadow_color, 10, cv2.LINE_AA)
            # Line 2: Foreground
            cv2.putText(overlay, text_l2, (base_x, base_y + line_height), font, font_scale_line2, text_foreground_color, 6, cv2.LINE_AA)

            cv2.addWeighted(overlay, ALPHA, frame, 1 - ALPHA, 0, frame)

    frames_out.append(frame)
    if f % max(1, (total_frames // 20)) == 0: print(f"Progress: {f*100//total_frames}%")

print(f"\n✅ 完了！ 処理時間: {time.time() - start_time:.2f}秒")

🚀 20秒ループ 生成開始...
Progress: 0%
Progress: 5%
Progress: 10%
Progress: 15%
Progress: 20%
Progress: 25%
Progress: 30%
Progress: 35%
Progress: 40%
Progress: 45%
Progress: 50%
Progress: 55%
Progress: 60%
Progress: 65%
Progress: 70%
Progress: 75%
Progress: 80%
Progress: 85%
Progress: 90%
Progress: 95%

✅ 完了！ 処理時間: 46.89秒


In [ ]:

import os
import time

# 保存パス（現在の設定を維持）
output_dir = "/content/drive/MyDrive/Colab Notebooks/output"
os.makedirs(output_dir, exist_ok=True)
temp_path = "/content/temp_raw.mp4"

timestamp = time.strftime("%Y%m%d_%H%M%S")
final_path = os.path.join(output_dir, f"sand_art_{DURATION}s_{timestamp}.mp4")

# 1. 一時ファイル生成
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_path, fourcc, FPS, (WIDTH, HEIGHT))

print("📦 映像データを生成中...")
for frame in frames_out:
    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
out.release()

# 2. ffmpegで爆速圧縮
# -preset ultrafast を追加して処理時間を大幅短縮
print("⚡ 爆速エンコード中（ultrafast設定）...")
!ffmpeg -y -i "{temp_path}" -vcodec libx264 -crf 23 -preset ultrafast -pix_fmt yuv420p -movflags +faststart "{final_path}" -loglevel error

# 一時ファイルの削除
if os.path.exists(temp_path):
    os.remove(temp_path)

# サイズ確認
file_size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\n✅ 完了！")
print(f"📂 保存先: {final_path}")
print(f"📦 サイズ: {file_size_mb:.2f} MB")

📦 映像データを生成中...
⚡ 爆速エンコード中（ultrafast設定）...
